# Обрезка праймеров — ERP003950

Обрезка технических праймеров для мышиного IgG heavy-chain датасета `ERP003950` (`Greiff 2014`).

Текущая политика: обрезаем только праймер constant-региона IgG, и только у того mate (R1/R2), где он реально сидит — маппинг per-sample ниже (`CONSTANT_PRIMER_MATE`). Второй mate копируется без изменений. V-region forward-праймеры сохраняются и не входят в FASTA для обрезки, чтобы не терять V/J-информативные основания.

Раньше эта логика жила в отдельном `scripts/primer_trim_mouse_constant_only.py`, который нужно было вручную закачивать в task workspace — встроено прямо сюда, ноутбук больше не зависит от внешнего файла.


In [ ]:
import os, sys, sysconfig, shutil, subprocess
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [_CONDA_ENV + "/lib/python3.11/site-packages", sysconfig.get_path("purelib")]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
print("cutadapt", shutil.which("cutadapt"))


In [ ]:
from pathlib import Path

THREADS = 1
ERROR_RATE = 0.2
MIN_OVERLAP = 10
FORWARD_ADAPTER = 'CARKGGATRRRCHGATGGGG'
REVERSE_ADAPTER = 'CCCCATCDGYYYATCCMYTG'

# Для каждого sample — какой mate (1=R1, 2=R2) реально несёт constant-region праймер.
CONSTANT_PRIMER_MATE = {
    'ERR346596': '1',
    'ERR346597': '1',
    'ERR346598': '1',
    'ERR346599': '2',
    'ERR346600': '1',
    'ERR346601': '1',
}


In [ ]:
def run_primer_trim_mouse(volume, dataset='ERP003950', force=False):
    if dataset != 'ERP003950':
        raise ValueError(f'This runner is configured only for ERP003950, got {dataset}')

    vol = Path(volume)
    src_dir = vol / 'results' / dataset / 'trimmed' / 'fastq'
    base = vol / 'results' / dataset / 'pr_trimmed'
    out_dir = base / 'fastq'
    logs_dir = base / 'cutadapt_logs'

    if force and base.exists():
        shutil.rmtree(base)

    out_dir.mkdir(parents=True, exist_ok=True)
    logs_dir.mkdir(parents=True, exist_ok=True)

    pairs = sorted(set(
        f.name.replace('_1.trim.fastq.gz', '').replace('_2.trim.fastq.gz', '')
        for f in src_dir.glob('*.trim.fastq.gz')
    ))
    print(f'[primer_trim_mouse] {dataset}: {len(pairs)} pairs')

    for sample in pairs:
        constant_mate = CONSTANT_PRIMER_MATE.get(sample)
        if constant_mate is None:
            raise RuntimeError(f'No constant-primer mate mapping for sample {sample}')
        for mate in ('1', '2'):
            src = src_dir / f'{sample}_{mate}.trim.fastq.gz'
            dest = out_dir / f'{sample}_{mate}.pr.fastq.gz'
            if not src.exists():
                raise FileNotFoundError(src)
            if dest.exists() and not force:
                print(f'  [{sample}_{mate}] already done, skip')
                continue
            if mate != constant_mate:
                shutil.copy2(src, dest)
                print(f'  [{sample}_{mate}] copied unchanged (non-constant mate)')
                continue
            cmd = [
                'cutadapt',
                '-j', str(THREADS),
                '--overlap', str(MIN_OVERLAP),
                '-e', str(ERROR_RATE),
                '--action=trim',
                '-g', FORWARD_ADAPTER,
                '-g', REVERSE_ADAPTER,
                '-o', str(dest),
                str(src),
            ]
            print(f'  [{sample}_{mate}] cutadapt constant-region trim ...')
            log_path = logs_dir / f'{sample}_{mate}.cutadapt.log'
            with open(log_path, 'a') as lf:
                res = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT)
            if res.returncode != 0:
                raise RuntimeError(f'cutadapt failed for {sample}_{mate}; see {log_path}')

    n = len(list(out_dir.glob('*.pr.fastq.gz')))
    print(f'[primer_trim_mouse] DONE: {n} pr-trimmed files')


### Запуск

Запускать после того, как `adapter_trim_mouse.ipynb` уже создал `results/ERP003950/trimmed/fastq`.


In [ ]:
run_primer_trim_mouse('/data/user/epishkin', 'ERP003950')
